# ConformalGuard: additional R-Judge experiments (Kaggle, results are kept)

Four experiments, each packed into `/kaggle/working/rjudge_extra_results.zip` as soon as it finishes:

1. **Calibrated LLM guards**: a local model, a Claude model and an OpenAI model score every record (whole record, and causally step by step); the scores are then calibrated conformally.
2. **Step-level guarantee** (Proposition 1) on the injection records, with step labels derived by code.
3. **In-domain recalibration and per-category calibration** under domain shift.

**Before running**

- Settings: Internet **on**, Accelerator **GPU T4 x2**.
- Add-ons → Secrets: add `ANTHROPIC_API_KEY` and `OPENAI_API_KEY` and attach both to this notebook. Never paste a key into a cell. A missing key just skips that judge.
- Set `OPENAI_MODEL` in the first code cell to a chat model your key can use.
- Then **Save Version → Save & Run All (Commit)**. Do not run interactively: Kaggle deletes the outputs of an interactive session. When the run finishes, open the saved version → Output → download `rjudge_extra_results.zip`.

The API judges are billed by the providers: about 2,000 requests per judge and view (571 records, 1,461 decision steps).

In [ ]:
CLAUDE_MODEL = "claude-opus-5-5"
CLAUDE_EFFORT = "medium"
OPENAI_MODEL = ""            # e.g. the id of a current chat model on your account; "" skips the OpenAI judge
LOCAL_MODEL = "Qwen/Qwen2.5-7B-Instruct"
SPLITS = 100                 # splits per protocol for the trained scorers

import os, subprocess
os.makedirs("/kaggle/temp", exist_ok=True)       # scratch space; only the zip goes to /kaggle/working
os.chdir("/kaggle/temp")
def sh(cmd):
    print("$", cmd, flush=True)
    return get_ipython().system(cmd)
sh("git clone -q --depth 1 -b claude/implement-conformalguard-UGndd https://github.com/rogerpanel/CV.git")
sh("git clone -q --depth 1 https://github.com/Lordog/R-Judge.git")
sh("git clone -q --depth 1 https://github.com/uiuc-kang-lab/InjecAgent.git")
sh("pip -q install sentence-transformers scikit-learn pytest accelerate anthropic openai")
os.chdir("/kaggle/temp/CV/ConformalGuard")
sh("git log --oneline -1")
RJ, IA = "../../R-Judge", "../../InjecAgent"

def pack():
    sh("cd /kaggle/temp/CV/ConformalGuard && zip -qr /kaggle/working/rjudge_extra_results.zip runs -x '*/cache/*'")
    sh("ls -la /kaggle/working/rjudge_extra_results.zip")

In [ ]:
# API keys come from Kaggle Secrets and are only placed in this process's environment.
try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
    for _k in ("ANTHROPIC_API_KEY", "OPENAI_API_KEY"):
        try:
            os.environ[_k] = _secrets.get_secret(_k)
            print(_k, "loaded")
        except Exception:
            print(_k, "not set: that judge will be skipped")
except ImportError:
    print("not on Kaggle: using keys already in the environment, if any")

## 0. Sanity check (all tests must pass)

In [ ]:
sh("python -m pytest tests/test_core.py tests/test_adaptive.py tests/test_rjudge.py tests/test_guards.py -q")

## 1a. Local model as a guard (GPU, one forward pass per unit)

In [ ]:
for mode in ("causal", "full"):
    sh(f"python scripts/rjudge_judge_local.py --rjudge-dir {RJ} --model {LOCAL_MODEL} --mode {mode} --out runs/guard_local_{mode}")
pack()

## 1b. Claude as a guard
Server-side refusal fallbacks are on, so a request declined by the safety classifiers is re-run on a fallback model; the output records which model served each row.

In [ ]:
if os.environ.get("ANTHROPIC_API_KEY"):
    for mode in ("causal", "full"):
        sh(f"python scripts/rjudge_judge_claude.py --rjudge-dir {RJ} --model {CLAUDE_MODEL} --effort {CLAUDE_EFFORT} --mode {mode} --out runs/guard_claude_{mode}")
        sh(f"python scripts/rjudge_judge_claude.py --rjudge-dir {RJ} --model {CLAUDE_MODEL} --effort {CLAUDE_EFFORT} --mode {mode} --out runs/guard_claude_{mode}")   # second pass retries failed calls
    pack()
else:
    print("skipped: no ANTHROPIC_API_KEY")

## 1c. OpenAI model as a guard

In [ ]:
if os.environ.get("OPENAI_API_KEY") and OPENAI_MODEL:
    for mode in ("causal", "full"):
        sh(f"python scripts/rjudge_judge_openai.py --rjudge-dir {RJ} --model {OPENAI_MODEL} --mode {mode} --out runs/guard_openai_{mode}")
        sh(f"python scripts/rjudge_judge_openai.py --rjudge-dir {RJ} --model {OPENAI_MODEL} --mode {mode} --out runs/guard_openai_{mode}")   # retries failed calls
    pack()
else:
    print("skipped: set OPENAI_MODEL and the OPENAI_API_KEY secret to run it")

## 1d. Calibrate every guard's scores (CPU, a few minutes)

In [ ]:
specs = " ".join(f"{name}_{mode}=runs/guard_{name}_{mode}/scores.jsonl"
                 for name in ("local", "claude", "openai") for mode in ("causal", "full")
                 if os.path.exists(f"runs/guard_{name}_{mode}/scores.jsonl"))
if specs:
    sh(f"python scripts/rjudge_guard_eval.py --rjudge-dir {RJ} --scores {specs} --reps 1000 --out runs/guard_eval")
    pack()

## 2. Step-level guarantee on the injection records

In [ ]:
sh(f"python scripts/rjudge_steps.py --rjudge-dir {RJ} --injecagent-dir {IA} --features st --reps {SPLITS} --out runs/rjudge_steps")
pack()

## 3. In-domain recalibration and per-category calibration

In [ ]:
sh(f"python scripts/rjudge_run.py --rjudge-dir {RJ} --features st --protocols random,recal --reps {SPLITS} --loco-reps 20 --out runs/rjudge_recal_minilm")
pack()